In [3]:
from pyspark.sql.functions import col

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CIC-IDS2017 Analysis") \
    .getOrCreate()

print("Spark started successfully")

c:\Users\nalin.karthik\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark started successfully


In [5]:
from pathlib import Path

# Locate the repository root so the notebook runs from any checkout location.
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "requirements.txt").exists() and (p / "src").is_dir()
)

path = str(PROJECT_ROOT / "MachineLearningCVE" / "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")

df = spark.read.csv(
    path,
    header=True,
    inferSchema=True
)

df = df.toDF(*[col.strip() for col in df.columns])

In [3]:
print("Total rows:", df.count())
print("Total columns:", len(df.columns))

Total rows: 225745
Total columns: 79


In [4]:
df.groupBy("Label") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

+------+------+
| Label| count|
+------+------+
|  DDoS|128027|
|BENIGN| 97718|
+------+------+



In [6]:
total = df.count()

df.groupBy("Label") \
    .count() \
    .withColumn("percentage", col("count") / total * 100) \
    .orderBy("count", ascending=False) \
    .show()

+------+------+------------------+
| Label| count|        percentage|
+------+------+------------------+
|  DDoS|128027|56.713105495138315|
|BENIGN| 97718| 43.28689450486168|
+------+------+------------------+



In [7]:
from pyspark.sql.functions import col, sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.show()

+----------------+-------------+-----------------+----------------------+---------------------------+---------------------------+---------------------+---------------------+----------------------+---------------------+---------------------+---------------------+----------------------+---------------------+------------+--------------+-------------+------------+------------+------------+-------------+------------+-----------+-----------+-----------+-------------+------------+-----------+-----------+-----------+-------------+-------------+-------------+-------------+-------------------+-----------------+-------------+-------------+-----------------+-----------------+------------------+-----------------+----------------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+-------------+-------------------+--------------------+--------------------+-------------------+------------------+--------------------+------------

In [8]:
from pyspark.sql.functions import col, isnan, sum

nan_counts = df.select([
    sum(isnan(col(c).cast("double")).cast("int")).alias(c)
    for c in df.columns
    if c != "Label"
])

nan_counts.show()


+----------------+-------------+-----------------+----------------------+---------------------------+---------------------------+---------------------+---------------------+----------------------+---------------------+---------------------+---------------------+----------------------+---------------------+------------+--------------+-------------+------------+------------+------------+-------------+------------+-----------+-----------+-----------+-------------+------------+-----------+-----------+-----------+-------------+-------------+-------------+-------------+-------------------+-----------------+-------------+-------------+-----------------+-----------------+------------------+-----------------+----------------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+-------------+-------------------+--------------------+--------------------+-------------------+------------------+--------------------+------------

In [ ]:
from pyspark.sql.functions import col, sum

inf_counts = df.select([
    sum(
        ((col(c) == float("inf")) | (col(c) == float("-inf"))).cast("int")
    ).alias(c)
    for c in df.columns
    if c != "Label"
])

inf_counts.show()

+----------------+-------------+-----------------+----------------------+---------------------------+---------------------------+---------------------+---------------------+----------------------+---------------------+---------------------+---------------------+----------------------+---------------------+------------+--------------+-------------+------------+------------+------------+-------------+------------+-----------+-----------+-----------+-------------+------------+-----------+-----------+-----------+-------------+-------------+-------------+-------------+-------------------+-----------------+-------------+-------------+-----------------+-----------------+------------------+-----------------+----------------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+-------------+-------------------+--------------------+--------------------+-------------------+------------------+--------------------+------------

: 

In [6]:
total_rows = df.count()
unique_rows = df.dropDuplicates().count()

print("Total rows:", total_rows)
print("Unique rows:", unique_rows)
print("Duplicate rows:", total_rows - unique_rows)

Total rows: 225745
Unique rows: 223112
Duplicate rows: 2633


In [7]:
df.describe().show()

+-------+-----------------+--------------------+------------------+----------------------+---------------------------+---------------------------+---------------------+---------------------+----------------------+---------------------+---------------------+---------------------+----------------------+---------------------+------------+--------------+------------------+------------------+--------------------+-----------------+--------------------+----------------+--------------------+--------------------+------------------+--------------------+-----------------+------------------+-------------------+------------------+-------------------+-------------+-------------+-------------+-------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+----------------------+--------------------+-------------------+--------------------+-------------------+-------------------+------------------+--------------+--

In [8]:
from pyspark.sql.functions import countDistinct

unique_counts = df.select([
    countDistinct(c).alias(c)
    for c in df.columns
])

unique_counts.show(truncate=False)

+----------------+-------------+-----------------+----------------------+---------------------------+---------------------------+---------------------+---------------------+----------------------+---------------------+---------------------+---------------------+----------------------+---------------------+------------+--------------+-------------+------------+------------+------------+-------------+------------+-----------+-----------+-----------+-------------+------------+-----------+-----------+-----------+-------------+-------------+-------------+-------------+-------------------+-----------------+-------------+-------------+-----------------+-----------------+------------------+-----------------+----------------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+-------------+-------------------+--------------------+--------------------+-------------------+------------------+--------------------+------------

In [9]:
constant_cols = [
    c for c in df.columns
    if df.select(c).distinct().count() == 1
]

print("Constant columns:")
print(constant_cols)

Constant columns:
['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']


In [10]:
constant_cols = [
    'Bwd PSH Flags',
    'Fwd URG Flags',
    'Bwd URG Flags',
    'CWE Flag Count',
    'Fwd Avg Bytes/Bulk',
    'Fwd Avg Packets/Bulk',
    'Fwd Avg Bulk Rate',
    'Bwd Avg Bytes/Bulk',
    'Bwd Avg Packets/Bulk',
    'Bwd Avg Bulk Rate'
]

df_analysis = df.drop(*constant_cols)

print("Original columns:", len(df.columns))
print("Remaining columns:", len(df_analysis.columns))

Original columns: 79
Remaining columns: 69


In [11]:
from pyspark.sql.functions import avg, col

df_analysis.groupBy("Label").agg(
    *[
        avg(col(c)).alias(c)
        for c in df_analysis.columns
        if c != "Label"
    ]
).show(truncate=False)

+------+-----------------+--------------------+-----------------+----------------------+---------------------------+---------------------------+---------------------+---------------------+----------------------+---------------------+---------------------+---------------------+----------------------+---------------------+------------+--------------+------------------+------------------+--------------------+------------------+-------------------+------------------+-----------------+--------------------+------------------+--------------------+------------------+------------------+-----------------+-----------------+-------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+----------------------+---------------------+-------------------+---------------------+-------------------+------------------+---------------------+---------------------+------------------+------------------

In [12]:
from pyspark.sql.functions import percentile_approx, col

numeric_cols = [
    c for c in df_analysis.columns
    if c != "Label"
]

median_df = df_analysis.groupBy("Label").agg(
    *[
        percentile_approx(col(c), 0.5).alias(c)
        for c in numeric_cols
    ]
)

median_df.show(truncate=False)

+------+----------------+-------------+-----------------+----------------------+---------------------------+---------------------------+---------------------+---------------------+----------------------+---------------------+---------------------+---------------------+----------------------+---------------------+------------+--------------+-------------+------------+------------+------------+-------------+------------+-----------+-----------+-----------+-------------+------------+-----------+-----------+-----------+-------------+-------------------+-----------------+-------------+-------------+-----------------+-----------------+------------------+-----------------+----------------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+-------------+-------------------+--------------------+--------------------+-------------------+-------------------+-----------------+-------------------+-----------------+----------------------+---

In [13]:
sample_df = df_analysis.sample(
    withReplacement=False,
    fraction=0.1,
    seed=42
)

pdf = sample_df.toPandas()

corr = pdf.drop(columns=["Label"]).corr()

c:\Users\nalin.karthik\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [14]:
import numpy as np

high_corr = []

for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        value = corr.iloc[i, j]

        if abs(value) >= 0.90:
            high_corr.append(
                (corr.columns[i], corr.columns[j], value)
            )

for pair in high_corr:
    print(pair)

('Flow Duration', 'Flow IAT Max', np.float64(0.9174019908143604))
('Flow Duration', 'Fwd IAT Total', np.float64(0.9968278012754975))
('Flow Duration', 'Fwd IAT Max', np.float64(0.9148831620066233))
('Flow Duration', 'Idle Max', np.float64(0.9156094593384708))
('Total Fwd Packets', 'Total Backward Packets', np.float64(0.9509039704025171))
('Total Fwd Packets', 'Total Length of Bwd Packets', np.float64(0.9352453267510267))
('Total Fwd Packets', 'Fwd Header Length34', np.float64(0.96876537245877))
('Total Fwd Packets', 'Bwd Header Length', np.float64(0.9291039074375484))
('Total Fwd Packets', 'Fwd Header Length55', np.float64(0.96876537245877))
('Total Fwd Packets', 'Subflow Fwd Packets', np.float64(1.0))
('Total Fwd Packets', 'Subflow Bwd Packets', np.float64(0.9509039704025171))
('Total Fwd Packets', 'Subflow Bwd Bytes', np.float64(0.9352453267510267))
('Total Backward Packets', 'Total Length of Bwd Packets', np.float64(0.9611092773321167))
('Total Backward Packets', 'Fwd Header Length3

In [16]:
text_features = [
    "Destination Port",
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Fwd Packet Length Mean",
    "Bwd Packet Length Mean",
    "Flow Packets/s",
    "Fwd Packets/s",
    "Bwd Packets/s",
    "Min Packet Length",
    "Max Packet Length",
    "Packet Length Mean",
    "Packet Length Std",
    "SYN Flag Count",
    "RST Flag Count",
    "PSH Flag Count",
    "ACK Flag Count",
    "Down/Up Ratio",
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward",
    "Active Mean",
    "Idle Mean"
]

In [17]:
from pyspark.sql.functions import col, isnan, sum

for c in text_features:
    invalid = df_analysis.filter(
        isnan(col(c).cast("double")) |
        (col(c) == float("inf")) |
        (col(c) == float("-inf"))
    ).count()

    if invalid > 0:
        print(c, "->", invalid)

Flow Packets/s -> 34


In [18]:
from pyspark.sql.functions import col, when

df_analysis = df_analysis.withColumn(
    "Flow Packets/s",
    when(
        (col("Flow Packets/s") == float("inf")) |
        (col("Flow Packets/s") == float("-inf")) |
        col("Flow Packets/s").isNull(),
        None
    ).otherwise(col("Flow Packets/s"))
)

In [19]:
df_analysis.filter(
    col("Flow Packets/s").isNull()
).count()

34

In [20]:
from pyspark.sql.functions import concat, lit, coalesce

df_rag = df_analysis.withColumn(
    "text_description",
    concat(
        lit("Traffic type: "), col("Label"),
        lit(". Destination port: "), col("Destination Port"),
        lit(". Flow duration: "), col("Flow Duration"),
        lit(". Forward packets: "), col("Total Fwd Packets"),
        lit(". Backward packets: "), col("Total Backward Packets"),
        lit(". Forward bytes: "), col("Total Length of Fwd Packets"),
        lit(". Backward bytes: "), col("Total Length of Bwd Packets"),
        lit(". Forward packet mean length: "), col("Fwd Packet Length Mean"),
        lit(". Backward packet mean length: "), col("Bwd Packet Length Mean"),
        lit(". Maximum packet length: "), col("Max Packet Length"),
        lit(". Packet length mean: "), col("Packet Length Mean"),
        lit(". SYN flags: "), col("SYN Flag Count"),
        lit(". RST flags: "), col("RST Flag Count"),
        lit(". PSH flags: "), col("PSH Flag Count"),
        lit(". ACK flags: "), col("ACK Flag Count"),
        lit(". Down/up ratio: "), col("Down/Up Ratio")
    )
)

In [21]:
df_rag.select("Label", "text_description").show(5, truncate=False)

+------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Label |text_description                                                                                                                                                                                                                                                                                                                               |
+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------